# Task 2 — Classifying power-quality events

**ELEC-E8131 · AI for Electrical Engineers · Colab lab, part 2 of 4**

### What you will learn about Colab
- reproducibility: why a seed is part of your result, not a detail
- caching generated data to Drive instead of regenerating it every time
- `%%time` and why it changes how you organise a notebook

### What you will learn about machine learning
- how a classifier differs from the regressor you built in Task 1
- why **accuracy is a dangerous metric** when classes are imbalanced
- how to read a confusion matrix, and why you should never report accuracy without one
- two different fixes for a rare class, and which one is actually better

### Setting
You are prototyping the classifier for a power-quality monitor on an 11 kV feeder. It captures
200 ms windows of the phase voltage at 5 kHz and must label each window as one of:

| class | what it is | how often |
|-------|-----------|-----------|
| `nominal` | clean 50 Hz | ~45 % |
| `sag` | voltage dips to 0.75–0.92 pu for part of the window | ~25 % |
| `harmonic` | 3rd, 5th, 7th harmonic content | ~25 % |
| `transient` | a short impulse, well under one cycle | **~5 %** |

That last row is the interesting one, and not only because it is rare.

---
## Part 1 — Generate the waveforms

We simulate the capture rather than downloading data. That is not a compromise: it means you know
the ground truth exactly, and you can change the physics and see what the classifier does about it.

It also means **the seed is part of your result**. Change the seed and every number in this
notebook moves. Report it alongside your results, always.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

FS = 5000.0                     # sampling rate [Hz]
WINDOW = 0.2                    # capture length [s]
N = int(FS * WINDOW)            # samples per window
t = np.arange(N) / FS
F0 = 50.0                       # fundamental [Hz]
CLASSES = ["nominal", "sag", "harmonic", "transient"]

print("%d samples per window, %.0f fundamental cycles" % (N, WINDOW * F0))

In [ ]:
def generate_dataset(n_total=3000, seed=1, noise=0.02):
    # Class shares: nominal 45 %, sag 25 %, harmonic 25 %, transient 5 %.
    rng = np.random.default_rng(seed)
    shares = [0.45, 0.25, 0.25, 0.05]
    counts = [int(round(s * n_total)) for s in shares]
    X, y = [], []

    for ci, (name, c) in enumerate(zip(CLASSES, counts)):
        for _ in range(c):
            amp = 1.0 + 0.05 * rng.standard_normal()      # supply voltage varies a little
            phase = rng.uniform(0, 2 * np.pi)             # capture is not synchronised
            v = amp * np.sin(2 * np.pi * F0 * t + phase)

            if name == "sag":
                dur = rng.uniform(0.04, 0.12)             # 2 to 6 cycles
                start = rng.uniform(0, WINDOW - dur)
                mask = (t >= start) & (t < start + dur)
                v[mask] *= rng.uniform(0.75, 0.92)        # depth of the dip [pu]

            elif name == "harmonic":
                for h in (3, 5, 7):
                    v += rng.uniform(0.01, 0.04) * np.sin(
                        2 * np.pi * h * F0 * t + rng.uniform(0, 2 * np.pi))

            elif name == "transient":
                dur = max(2, int(rng.uniform(0.0004, 0.0012) * FS))   # 0.4 to 1.2 ms
                start = rng.integers(0, N - dur)
                v[start:start + dur] += rng.uniform(0.10, 0.25) * rng.choice([-1, 1])

            v += noise * rng.standard_normal(N)           # measurement noise
            X.append(v)
            y.append(ci)

    X = np.array(X, dtype=np.float32)
    y = np.array(y)
    perm = rng.permutation(len(y))                        # never leave the data sorted by class
    return X[perm], y[perm]

In [ ]:
%%time
X, y = generate_dataset(n_total=3000, seed=1)

print("X:", X.shape, X.dtype, "  %.1f MB" % (X.nbytes / 1e6))
for ci, name in enumerate(CLASSES):
    print("  %-10s %4d windows  (%.1f %%)" % (name, (y == ci).sum(), 100 * (y == ci).mean()))

Note the transient count. **About 150 examples out of 3000.** Hold that number in your head; it
comes back in Part 5.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
for ci, (ax, name) in enumerate(zip(axes, CLASSES)):
    i = np.where(y == ci)[0][0]
    ax.plot(t * 1e3, X[i], lw=.8)
    ax.set_ylabel("v [pu]")
    ax.set_title(name, loc="left", fontsize=10)
    ax.grid(alpha=.3)
axes[-1].set_xlabel("time [ms]")
plt.tight_layout(); plt.show()

**Look carefully at the transient panel before moving on.** Can you see the impulse? Zoom in by
re-plotting a narrow time range if you cannot. The impulse is 0.10–0.25 pu, under one millisecond,
sitting on top of 2 % measurement noise.

This is not an artificial difficulty. Sub-cycle transients from switching events are genuinely
small and genuinely brief, and they are exactly what a monitor is installed to catch.

---
## Part 2 — Features, not raw samples

Each window is 1000 numbers. You *could* feed all 1000 to a network. You should not, yet: with
3000 examples and 1000 inputs the first layer alone would have more parameters than you have data
points, and you already know from Task 1 what happens when capacity outruns data.

Instead compute a handful of quantities that a power engineer would compute anyway:

| feature | definition | which class it is for |
|---------|-----------|----------------------|
| `rms` | $\sqrt{\overline{v^2}}$ | overall level |
| `crest` | $\max|v| / v_{rms}$ | peaky waveforms |
| `thd` | harmonic bins over fundamental bin | `harmonic` |
| `dip` | lowest half-cycle RMS, relative to window RMS | `sag` |

This is **feature engineering**, and in Task 1 you already saw its power: the $\log_{10}$ transform
did more for the diode fit than any architecture change.

In [ ]:
def extract_features(X):
    rms = np.sqrt(np.mean(X ** 2, axis=1))
    crest = np.max(np.abs(X), axis=1) / rms

    spec = np.abs(np.fft.rfft(X, axis=1))
    freqs = np.fft.rfftfreq(X.shape[1], 1 / FS)
    bin_at = lambda f: spec[:, np.argmin(np.abs(freqs - f))]
    thd = np.sqrt(bin_at(150) ** 2 + bin_at(250) ** 2 + bin_at(350) ** 2) / bin_at(50.0)

    k = X.shape[1] // 20                       # 20 sub-windows = half a cycle each
    sub = X[:, :k * 20].reshape(len(X), 20, k)
    dip = np.sqrt(np.mean(sub ** 2, axis=2)).min(axis=1) / rms

    return np.column_stack([rms, crest, thd, dip])


FEATURE_NAMES = ["rms", "crest", "thd", "dip"]
F = extract_features(X)
print("features:", F.shape)

print("\nclass means:")
print("%-10s %8s %8s %8s %8s" % ("", *FEATURE_NAMES))
for ci, name in enumerate(CLASSES):
    print("%-10s %8.3f %8.3f %8.3f %8.3f" % (name, *F[y == ci].mean(axis=0)))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for j, (ax, fname) in enumerate(zip(axes, FEATURE_NAMES)):
    for ci, cname in enumerate(CLASSES):
        ax.hist(F[y == ci, j], bins=40, alpha=.55, label=cname, density=True)
    ax.set_title(fname); ax.grid(alpha=.3)
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

**Question before you continue.** From the class means and these histograms, predict which classes
the model will separate easily and which it will confuse. Write your prediction into a text cell
now. You will check it against the confusion matrix in Part 5.

---
## Part 3 — Cache the data

You just timed the generation. It is not slow, but it is not free either, and you will restart this
runtime several times before the lab is over. Regenerating derived data on every restart is one of
the most common ways to waste time in Colab.

Save the arrays to Drive once, and load them thereafter.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

OUT = "/content/drive/MyDrive/elec_e8131_colab/task2"
os.makedirs(OUT, exist_ok=True)
CACHE = os.path.join(OUT, "pq_dataset_seed1.npz")

# np.savez_compressed writes several arrays into one file.
np.savez_compressed(CACHE, X=X, y=y, seed=1, noise=0.02)
print("%.1f MB written to %s" % (os.path.getsize(CACHE) / 1e6, CACHE))

In [ ]:
%%time
# From now on, this is how you would start the notebook.
blob = np.load(CACHE)
X, y = blob["X"], blob["y"]
print("loaded", X.shape, "seed", blob["seed"])

Compare this timing with the generation cell in Part 1.

Notice that the seed went **into the file**. A cached dataset with no record of how it was produced
is a liability: six weeks from now you will not remember, and neither will the person reviewing
your work.

---
## Part 4 — Train a classifier

Three differences from the regressor in Task 1:

1. The output layer has **4 units**, one score per class, not 1.
2. The loss is `CrossEntropyLoss`, not MSE. It takes raw scores (logits) and integer labels —
   do **not** put a softmax at the end of the model, `CrossEntropyLoss` applies it internally.
3. We split **three** ways. Train fits the weights; validation is where you make decisions;
   test is touched once, at the very end. If you tune against the test set you no longer have an
   estimate of anything.

In [ ]:
rng = np.random.default_rng(7)
idx = rng.permutation(len(y))
n_tr, n_va = int(0.6 * len(y)), int(0.2 * len(y))
tr_idx = idx[:n_tr]
va_idx = idx[n_tr:n_tr + n_va]
te_idx = idx[n_tr + n_va:]

# Standardise using TRAINING statistics only. Using the full dataset here would leak.
mu, sd = F[tr_idx].mean(axis=0), F[tr_idx].std(axis=0)
Fs = (F - mu) / sd

to_x = lambda i: torch.tensor(Fs[i], dtype=torch.float32)
to_y = lambda i: torch.tensor(y[i], dtype=torch.long)

print("train %d | val %d | test %d" % (len(tr_idx), len(va_idx), len(te_idx)))
print("transients in test set:", (y[te_idx] == 3).sum())

In [ ]:
def train_classifier(n_features, class_weight=None, epochs=500, lr=1e-2, seed=0):
    torch.manual_seed(seed)
    model = nn.Sequential(
        nn.Linear(n_features, 16), nn.ReLU(),
        nn.Linear(16, 16), nn.ReLU(),
        nn.Linear(16, len(CLASSES)),          # 4 logits, no softmax here
    )
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss(weight=class_weight)

    xtr, ytr = to_x(tr_idx)[:, :n_features], to_y(tr_idx)
    xva, yva = to_x(va_idx)[:, :n_features], to_y(va_idx)

    hist = []
    for epoch in range(epochs):
        opt.zero_grad()
        loss = loss_fn(model(xtr), ytr)
        loss.backward()
        opt.step()
        if epoch % 10 == 0:
            with torch.no_grad():
                va_acc = (model(xva).argmax(1) == yva).float().mean().item()
            hist.append((epoch, loss.item(), va_acc))
    return model, np.array(hist)


def evaluate(model, which, n_features):
    with torch.no_grad():
        pred = model(to_x(which)[:, :n_features]).argmax(1).numpy()
    true = y[which]
    cm = np.zeros((len(CLASSES), len(CLASSES)), dtype=int)
    for a, b in zip(true, pred):
        cm[a, b] += 1
    acc = (pred == true).mean()
    recall = cm.diagonal() / cm.sum(axis=1)
    return acc, recall, cm

In [ ]:
model_plain, hist_plain = train_classifier(n_features=4)

acc, recall, cm = evaluate(model_plain, va_idx, 4)
print("validation accuracy: %.1f %%" % (100 * acc))

About 96 % accuracy from four hand-computed features and a network with a few hundred parameters.

Write down whether you would be happy to ship this. Then continue.

---
## Part 5 — The confusion matrix

Accuracy is a single number summarising a 4×4 table. Summarising throws information away, and here
it throws away the only information that matters.

In [ ]:
def show_confusion(cm, title=""):
    fig, ax = plt.subplots(figsize=(4.6, 4.2))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(CLASSES)), CLASSES, rotation=45, ha="right")
    ax.set_yticks(range(len(CLASSES)), CLASSES)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(title)
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.tight_layout(); plt.show()


acc, recall, cm = evaluate(model_plain, va_idx, 4)
show_confusion(cm, "4 features, unweighted loss")
print("accuracy: %.1f %%\n" % (100 * acc))
for name, r, n in zip(CLASSES, recall, cm.sum(axis=1)):
    print("  %-10s recall %.2f   (%d in validation set)" % (name, r, n))

### Read the bottom row

`nominal`, `sag` and `harmonic` are essentially perfect. `transient` recall is about **0.34** —
the monitor would miss roughly two out of every three switching transients, which is the one event
class you installed it to catch.

And the accuracy is still 96 %, because transients are 5 % of the data. **A classifier that
ignored transients completely would score about 95 %.** Your model is barely beating that, and
accuracy cannot tell you so.

### Why the optimiser did this

It is not a bug. Cross-entropy sums over samples, so each of the ~90 transients in the training set
contributes as much as each of the ~810 nominal windows. Learning to detect a faint, brief impulse
would cost some accuracy on the common classes; the loss says that is a bad trade. The optimiser is
correctly minimising the objective you gave it. **The objective is wrong.**

This is the same failure mode as Task 1, Part 4. There the loss was dominated by large-current
samples; here it is dominated by common-class samples. In both cases the network was fine and the
loss function was measuring the wrong thing.

> Check the prediction you wrote in Part 2. Did you anticipate this?

---
## Part 6 — Fix 1: reweight the loss

The direct fix is to tell the loss that a missed transient costs more. `CrossEntropyLoss` takes a
per-class `weight`. The standard choice is inverse frequency, so each *class* contributes equally
rather than each *sample*.

In [ ]:
counts = np.bincount(y[tr_idx], minlength=len(CLASSES))
w = len(y[tr_idx]) / (len(CLASSES) * counts)
class_weight = torch.tensor(w, dtype=torch.float32)

for name, c, wi in zip(CLASSES, counts, w):
    print("  %-10s %4d training samples -> weight %.2f" % (name, c, wi))

model_weighted, hist_weighted = train_classifier(n_features=4, class_weight=class_weight)
acc_w, recall_w, cm_w = evaluate(model_weighted, va_idx, 4)
show_confusion(cm_w, "4 features, weighted loss")
print("accuracy: %.1f %%" % (100 * acc_w))
for name, r in zip(CLASSES, recall_w):
    print("  %-10s recall %.2f" % (name, r))

Transient recall roughly doubles, to about 0.69. **Overall accuracy falls**, to about 92 %.

Both of those are correct and expected. The model now guesses `transient` more readily, which
catches more real ones and also mislabels some `nominal` windows. Because `nominal` is common,
a few percent of it costs more accuracy than all the recovered transients gain.

If your instinct is that accuracy going down means the model got worse, this is the moment to
revise it. **A metric is a statement about what you care about.** For a protection relay, a missed
transient and a spurious alarm are not remotely equivalent costs, and no single accuracy figure
encodes that.

---
## Part 7 — Fix 2: give it a feature that can actually see the event

Reweighting made the model try harder with the information it had. But look back at the four
features and ask: **which of them can even represent a 0.6 ms impulse?**

- `rms` averages over 1000 samples — a sub-millisecond blip barely moves it.
- `dip` takes a minimum over half-cycle windows — designed for sags, blind to impulses.
- `thd` looks at three specific harmonic bins — an impulse is broadband, not harmonic.
- `crest` is the only one that responds at all, and it is easily masked by noise.

The physically right detector for an impulse is the **slew rate**. A 50 Hz sinusoid changes slowly
between samples; a step edge does not.

In [ ]:
def extract_features_v2(X):
    base = extract_features(X)
    rms = np.sqrt(np.mean(X ** 2, axis=1))
    slew = np.abs(np.diff(X, axis=1)).max(axis=1) / rms     # largest sample-to-sample jump
    return np.column_stack([base, slew])


FEATURE_NAMES_V2 = FEATURE_NAMES + ["slew"]
F2 = extract_features_v2(X)

mu2, sd2 = F2[tr_idx].mean(axis=0), F2[tr_idx].std(axis=0)
Fs = (F2 - mu2) / sd2          # to_x() reads Fs, so the helpers now see 5 features

plt.figure(figsize=(5, 3))
for ci, cname in enumerate(CLASSES):
    plt.hist(F2[y == ci, 4], bins=40, alpha=.55, label=cname, density=True)
plt.xlabel("slew"); plt.legend(fontsize=8); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
model_v2, hist_v2 = train_classifier(n_features=5)
acc2, recall2, cm2 = evaluate(model_v2, va_idx, 5)
show_confusion(cm2, "5 features, unweighted loss")
print("accuracy: %.1f %%" % (100 * acc2))
for name, r in zip(CLASSES, recall2):
    print("  %-10s recall %.2f" % (name, r))

About **98 % accuracy and 0.72 transient recall**, from an *unweighted* loss.

Run the cell below to add the fourth combination, then compare all four on validation data:

| model | accuracy | transient recall (n≈32) |
|-------|----------|-------------------------|
| 4 features, unweighted | ~0.96 | ~0.34 |
| 4 features, weighted | ~0.91 | ~0.69 |
| 5 features, unweighted | ~0.98 | ~0.72 |
| 5 features, weighted | ~0.97 | ~0.78 |

Read this carefully, because the honest reading is more interesting than the tidy one.

Reweighting bought recall by **giving up five points of accuracy** — a trade, and a defensible one.
The extra feature bought the *same* recall for **free**: 0.72 against 0.69 is well inside the
uncertainty of a recall estimated from 32 samples, but the accuracy difference (0.98 against 0.91)
is not. Combining both is best of all.

So the feature did not magically beat the weighting on recall. What it did was remove the
**trade-off**. The 4-feature model could only find more transients by becoming trigger-happy about
`nominal`, because it had no input distinguishing the two. The 5-feature model does not face that
dilemma.

**The lesson, and it is the most transferable thing in this lab:** when a model underperforms on a
subgroup, ask first whether the information needed to get that subgroup right is present in the
input at all. Loss weighting, resampling, and bigger architectures are all ways of squeezing an
input representation harder. None of them can add information that is not there.

### A word about those recall numbers

Your validation set contains only about 32 transients. A recall estimated from 32 samples has a
standard error of roughly **±0.08**. Two models differing by 0.05 in recall are not distinguishable
with this much data, and you should not claim otherwise.

Always report the count alongside the rate. `recall 0.72` is a number; `recall 0.72 (23/32)` is
evidence.

In [ ]:
model_both, _ = train_classifier(n_features=5, class_weight=class_weight)
acc_b, recall_b, cm_b = evaluate(model_both, va_idx, 5)
print("5 features + weights: accuracy %.3f, transient recall %.2f (n=%d)"
      % (acc_b, recall_b[3], cm_b[3].sum()))

---
## Part 8 — Decide, then touch the test set

Everything above was measured on **validation** data. You have now compared four candidate models.
Pick one — on validation evidence and on your engineering judgement about the application — and
evaluate it once on the test set.

Once you have looked at the test set, you are done. Going back to tune further makes the test
number meaningless.

In [ ]:
FINAL_MODEL = model_v2      # <- change this to your choice
N_FEATURES = 5              # <- and this to match

acc_t, recall_t, cm_t = evaluate(FINAL_MODEL, te_idx, N_FEATURES)
show_confusion(cm_t, "TEST SET - final evaluation")
print("test accuracy: %.1f %%" % (100 * acc_t))
for name, r, n in zip(CLASSES, recall_t, cm_t.sum(axis=1)):
    print("  %-10s recall %.2f  (n=%d)" % (name, r, n))

In [ ]:
torch.save({
    "state_dict": FINAL_MODEL.state_dict(),
    "n_features": N_FEATURES,
    "feature_names": FEATURE_NAMES_V2[:N_FEATURES],
    "mu": mu2[:N_FEATURES].tolist(),
    "sd": sd2[:N_FEATURES].tolist(),
    "classes": CLASSES,
    "seed": 1,
}, os.path.join(OUT, "pq_classifier.pt"))
print("saved")

---
## Hand-in

Share the notebook (**Share → Anyone with the link → Viewer**) with your answers in text cells.

1. A classifier that always predicts `nominal` would score 45 %. One that never predicts
   `transient` but is otherwise perfect scores 95 %. What does that tell you about using accuracy
   as the headline number for this application?
2. In Part 6, accuracy went **down** and you were told this was an improvement. Justify that
   claim in terms of the cost of the two error types on a real feeder.
3. Explain why the `slew` feature helped more than the class weights did. Your answer should refer
   to the physics of the transient, not to the training procedure.
4. Which model did you take to the test set, and why? Was your test-set result close to your
   validation result? If not, what would explain the gap?
5. You reported a transient recall from a validation set containing a few dozen transients.
   How much confidence does that number deserve, and what would you do about it?

### Optional extensions

- **Do it without features.** Feed the 1000 raw samples to an MLP and compare. You will need far
  more data, or far more regularisation, or both. This is the honest motivation for convolutional
  networks, which you would reach for next.
- **Make it harder.** Raise `noise` to 0.04 in `generate_dataset` and re-run. At what noise level
  does the slew feature stop working, and does that match your estimate of the impulse SNR?
- **Add a fifth class.** Voltage swell (amplitude rises to 1.1–1.2 pu). Which existing feature
  already detects it, and does adding the class hurt the others?